# scWAT Xenium section QC: Phases 0-2

Run this notebook once per section. It retains every cell and preserves raw sparse counts.

## Goal

Validate one Xenium section, reconcile its panel, import sparse counts, calculate section-specific QC flags, and write a reload-validated artifact bundle.

## Setup

### Parameters

Change `REGION_ID` for manual execution. Launchers inject the same parameters without editing the source notebook.

In [ ]:
PROJECT_ROOT <- "/dssg/home/acct-svetoslav_chakarov/svetoslav_chakarov/Lab_members/Yanan_Hu"
PIPELINE_REPO <- file.path(PROJECT_ROOT, "adipose_analysis", "YNH_Xenium_scWAT")
INPUT_ROOT <- file.path(PROJECT_ROOT, "adipose_data")
REGION_ID <- "Region_1"
RUN_LABEL <- "full_notebook_qc_v1"
METADATA_PATH <- file.path(PIPELINE_REPO, "config", "scwat_sample_manifest.tsv")
EXPECTED_SECTION_COUNT <- 4L
SEED <- 20260814L
STRICT_MODE <- FALSE


In [ ]:
OUTPUT_ROOT <- file.path(PROJECT_ROOT, "adipose_analysis", "scwat_qc_outputs", RUN_LABEL)
SECTION_OUTPUT_DIR <- file.path(OUTPUT_ROOT, "sections", REGION_ID)
source(file.path(PIPELINE_REPO, "R", "source.R"))
for (package in c("Matrix", "jsonlite", "ggplot2")) require_package(package)
validate_runtime_paths(PROJECT_ROOT, INPUT_ROOT, SECTION_OUTPUT_DIR, tempdir())
dir.create(SECTION_OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)
stopifnot(REGION_ID %in% paste0("Region_", seq_len(EXPECTED_SECTION_COUNT)))
cat("Section:", REGION_ID, "\nOutput:", SECTION_OUTPUT_DIR, "\n")


## Inputs

The selected raw/subset section directory is discovered by region ID. All raw files remain read-only.

In [ ]:
all_sections <- discover_xenium_sections(INPUT_ROOT, EXPECTED_SECTION_COUNT)
selected_section <- discover_one_section(INPUT_ROOT, REGION_ID)
selected_section


## Phase 0 - Configuration and metadata contract

In [ ]:
if (nzchar(METADATA_PATH)) {
  assert_path_within(PROJECT_ROOT, METADATA_PATH)
  full_manifest <- utils::read.delim(METADATA_PATH, check.names = FALSE)
} else {
  full_manifest <- create_synthetic_manifest(all_sections$region_id, SEED)
}
manifest_check <- validate_sample_manifest(full_manifest, all_sections$region_id)
if (!manifest_check$valid) stop(paste(manifest_check$issues, collapse = "; "))
section_manifest <- full_manifest[full_manifest$region_id == REGION_ID, , drop = FALSE]
section_manifest$region_dir <- selected_section$region_dir
configuration <- data.frame(
  key = c("project_root", "pipeline_repo", "input_root", "output_root", "region_id", "run_label", "seed", "strict_mode"),
  value = c(PROJECT_ROOT, PIPELINE_REPO, INPUT_ROOT, OUTPUT_ROOT, REGION_ID, RUN_LABEL, SEED, STRICT_MODE)
)
environment <- data.frame(
  item = c("R_version", "platform", "tempdir", "Matrix", "jsonlite", "ggplot2"),
  value = c(R.version.string, R.version$platform, tempdir(), as.character(packageVersion("Matrix")), as.character(packageVersion("jsonlite")), as.character(packageVersion("ggplot2")))
)
section_manifest


## Phase 1 - Provenance, integrity, panel reconciliation, and QC gate

In [ ]:
region_dir <- selected_section$region_dir[[1]]
inventory <- inventory_section_files(region_dir, REGION_ID, calculate_md5 = TRUE)
if (!all(inventory$exists)) stop(paste("Missing required files:", paste(inventory$relative_path[!inventory$exists], collapse = ", ")))
integrity <- validate_section_integrity(region_dir, REGION_ID)
alarms <- extract_analysis_alarms(file.path(region_dir, "analysis_summary.html"))
signature <- utils::read.delim(file.path(PIPELINE_REPO, "config", "eos_gene_sets.tsv"), check.names = FALSE)
installed_genes <- read_custom_panel_genes(file.path(region_dir, "gene_panel.json"))
gene_sets <- setNames(signature$gene_set, signature$gene)
panel_reconciliation <- reconcile_panel(signature$gene, installed_genes, gene_sets)
features_preview <- read_xenium_features(file.path(region_dir, "cell_feature_matrix", "features.tsv.gz"))
feature_type_summary <- as.data.frame(table(features_preview$feature_type), stringsAsFactors = FALSE)
names(feature_type_summary) <- c("feature_type", "n_features")
list(integrity = integrity, alarms = alarms, panel = table(panel_reconciliation$status))


## Phase 2 - Sparse import and section-specific cell QC

In [ ]:
xenium <- import_xenium_mex(region_dir)
qc <- calculate_xenium_cell_qc(xenium$counts, xenium$cells, REGION_ID)
metadata_columns <- intersect(c("mouse_id", "side", "section_id", "biological_replicate_id", "genotype", "treatment", "condition", "age_weeks", "metadata_status", "do_not_interpret"), names(section_manifest))
for (column in metadata_columns) qc$cell_metadata[[column]] <- section_manifest[[column]][[1]]
qc$summary


In [ ]:
section_plots <- plot_section_qc(qc$cell_metadata, REGION_ID)
print(section_plots$counts)
print(section_plots$features)
print(section_plots$area)
print(section_plots$spatial)


## Checks

Write every required artifact, then reload the saved sparse object and verify dimensions and cell alignment.

In [ ]:
artifact_paths <- write_section_artifacts(
  project_root = PROJECT_ROOT, output_dir = SECTION_OUTPUT_DIR, region_id = REGION_ID,
  configuration = configuration, manifest = section_manifest, environment = environment,
  inventory = inventory, integrity = integrity, feature_type_summary = feature_type_summary,
  panel_reconciliation = panel_reconciliation, alarms = alarms, qc = qc,
  counts = xenium$counts, features = xenium$features, strict_mode = STRICT_MODE
)
stopifnot(validate_section_artifacts(SECTION_OUTPUT_DIR, REGION_ID))
readiness <- utils::read.delim(file.path(SECTION_OUTPUT_DIR, "section_readiness_gates.tsv"), check.names = FALSE)
readiness


## Outputs

All outputs are section-specific. A `HOLD` or `PENDING` gate blocks biological interpretation but preserves diagnostic QC.

In [ ]:
data.frame(artifact = basename(artifact_paths), path = artifact_paths)[seq_len(min(length(artifact_paths), 20L)), , drop = FALSE]
cat("Completed", REGION_ID, "with", nrow(qc$cell_metadata), "cells; zero cells deleted.\n")
